In [1]:
pip install transformers datasets

Note: you may need to restart the kernel to use updated packages.


In [45]:
from transformers import AutoImageProcessor, ViTForImageClassification
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
from datasets import load_dataset
from copy import deepcopy
from PIL import Image
import time
import numpy as np
import random

In [46]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [145]:
dataset = load_dataset("huggingface/cats-image", trust_remote_code=True)
image = dataset["test"]["image"][0]

image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224") #will do processing of images: resize , normalize, divide into patches etc
pretrained_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

inputs = image_processor(image, return_tensors="pt") #return the processes image as a pt(pytorch) tensor

with torch.no_grad():                     # extracting the confidence score that the images belong to which class accroding to the pretrained mode (no gradient calculation)
    logits = pretrained_model(**inputs).logits

# pretrained_model predicts one of the 1000 ImageNet classes
predicted_label = logits.argmax(-1).item()
print(pretrained_model.config.id2label[predicted_label])

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Egyptian cat


In [146]:
pretrained_model.vit.encoder.layer[5].attention

ViTSdpaAttention(
  (attention): ViTSdpaSelfAttention(
    (query): Linear(in_features=768, out_features=768, bias=True)
    (key): Linear(in_features=768, out_features=768, bias=True)
    (value): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (output): ViTSelfOutput(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)

In [147]:
pretrained_model.vit.encoder.layer[5].attention.attention.query.weight

Parameter containing:
tensor([[-0.0595,  0.0012, -0.0924,  ..., -0.0064, -0.0817, -0.1555],
        [ 0.0100, -0.2533, -0.1143,  ..., -0.0600,  0.0988,  0.1554],
        [ 0.0221, -0.0878,  0.0436,  ..., -0.0568, -0.0363,  0.0352],
        ...,
        [ 0.1537,  0.1259, -0.0712,  ..., -0.0925,  0.0327, -0.0702],
        [-0.0553, -0.0606, -0.0897,  ...,  0.0685,  0.0981,  0.2262],
        [ 0.0309,  0.0175, -0.1598,  ..., -0.0682,  0.0207, -0.0420]],
       requires_grad=True)

In [148]:
pretrained_model.vit.encoder.layer[0].attention.attention.query

Linear(in_features=768, out_features=768, bias=True)

In [149]:
class LoRALayer():
    def __init__(
        self,
        r: int,
        lora_alpha: int,
        lora_dropout: float,
        merge_weights: bool,
    ):
        self.r = r
        self.lora_alpha = lora_alpha
        # Optional dropout
        if lora_dropout > 0.:
            self.lora_dropout = nn.Dropout(p=lora_dropout)
        else:
            self.lora_dropout = lambda x:x
        # Mark the weight as unmerged
        self.merged = False
        self.merge_weights = merge_weights


class LinearLoRA(nn.Linear, LoRALayer):
    def __init__(self,
                 existing_linear: nn.Linear,
                 r: int = 4,
                 lora_alpha: int = 1,
                 lora_dropout: float = 0.,
                 fan_in_fan_out: bool = True,
                 merge_weights: bool = True
                 ):

        super().__init__(
                         in_features = existing_linear.in_features,
                         out_features = existing_linear.out_features
                         )

        LoRALayer.__init__(self,
                           r=r,
                           lora_alpha = lora_alpha,
                           lora_dropout = lora_dropout,
                           merge_weights = merge_weights
                           )
        self.fan_in_fan_out = fan_in_fan_out
        
        self.weight.data = existing_linear.weight.data.clone()
        if existing_linear.bias is not None:
            self.bias.data = existing_linear.bias.data.clone()

        # Actual trainable parameters
        if r > 0:
            self.lora_A = nn.Parameter(self.weight.new_zeros((r, existing_linear.in_features)))
            self.lora_B = nn.Parameter(self.weight.new_zeros((existing_linear.out_features, r)))
            self.scaling = self.lora_alpha / self.r
        
        # self.reset_parameters()
        if fan_in_fan_out:
            self.weight.data = self.weight.data.transpose(0, 1)

    def reset_parameters(self):
        nn.Linear.reset_parameters(self)
        if hasattr(self, 'lora_A'):
            # initialize A the same way as the default for nn.Linear and B to zero
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def train(self, mode: bool = True):
        def T(w):
            return w.transpose(0, 1) if self.fan_in_fan_out else w
        nn.Linear.train(self, mode)
        if mode:    # If we are in the training mode (model.train())
            if self.merge_weights and self.merged:
                # Make sure that the weights are not merged
                if self.r > 0:
                    self.weight.data -= T(self.lora_B @ self.lora_A) * self.scaling
                self.merged = False
        else:       # If we are in the inference/eval mode (model.eval())
            if self.merge_weights and not self.merged:
                # Merge the weights and mark it
                if self.r > 0:
                    self.weight.data += T(self.lora_B @ self.lora_A) * self.scaling
                self.merged = True

    def forward(self, x: torch.Tensor):
            def T(w):
                return w.transpose(0, 1) if self.fan_in_fan_out else w
            if self.r > 0 and not self.merged:
                result = F.linear(x, T(self.weight), bias=self.bias)
                result += (self.lora_dropout(x) @ self.lora_A.transpose(0, 1) @ self.lora_B.transpose(0, 1)) * self.scaling
                return result
            else:
                return F.linear(x, T(self.weight), bias=self.bias)


In [150]:
def replace_attention_layers(model):
    for layer in model.vit.encoder.layer:

        # Replace query, key, and value layers with LoRA
        layer.attention.attention.query = LinearLoRA(layer.attention.attention.query)
        layer.attention.attention.key = LinearLoRA(layer.attention.attention.key)
        layer.attention.attention.value = LinearLoRA(layer.attention.attention.value)

    return model

# Apply to your pretrained model
model = replace_attention_layers(pretrained_model)

In [151]:
model.vit.encoder.layer[5].attention.attention.query.weight

Parameter containing:
tensor([[-0.0595,  0.0100,  0.0221,  ...,  0.1537, -0.0553,  0.0309],
        [ 0.0012, -0.2533, -0.0878,  ...,  0.1259, -0.0606,  0.0175],
        [-0.0924, -0.1143,  0.0436,  ..., -0.0712, -0.0897, -0.1598],
        ...,
        [-0.0064, -0.0600, -0.0568,  ..., -0.0925,  0.0685, -0.0682],
        [-0.0817,  0.0988, -0.0363,  ...,  0.0327,  0.0981,  0.0207],
        [-0.1555,  0.1554,  0.0352,  ..., -0.0702,  0.2262, -0.0420]],
       requires_grad=True)

In [152]:
model.vit.encoder.layer[5].attention

ViTSdpaAttention(
  (attention): ViTSdpaSelfAttention(
    (query): LinearLoRA(in_features=768, out_features=768, bias=True)
    (key): LinearLoRA(in_features=768, out_features=768, bias=True)
    (value): LinearLoRA(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (output): ViTSelfOutput(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)

In [153]:
def freeze_params_except_lora_and_classifier(model):

  for param in model.parameters():
      param.requires_grad_(False)

  for name, param in model.named_parameters():
      if 'lora_A' in name or 'lora_B' in name:
          param.requires_grad_(True)

  model.classifier.requires_grad_(True)

def change_classifier(model):

  model.classifier = nn.Linear(
  in_features=768,  # ViT hidden size
  out_features=10   # CIFAR-10 classes
  )

change_classifier(model)
freeze_params_except_lora_and_classifier(model)

In [154]:
#### Verifying frozen params
def verify_params(model, trainable:bool):
  for name, param in model.named_parameters():
    if param.requires_grad == trainable:
      print(name)

verify_params(model, trainable=True)

vit.encoder.layer.0.attention.attention.query.lora_A
vit.encoder.layer.0.attention.attention.query.lora_B
vit.encoder.layer.0.attention.attention.key.lora_A
vit.encoder.layer.0.attention.attention.key.lora_B
vit.encoder.layer.0.attention.attention.value.lora_A
vit.encoder.layer.0.attention.attention.value.lora_B
vit.encoder.layer.1.attention.attention.query.lora_A
vit.encoder.layer.1.attention.attention.query.lora_B
vit.encoder.layer.1.attention.attention.key.lora_A
vit.encoder.layer.1.attention.attention.key.lora_B
vit.encoder.layer.1.attention.attention.value.lora_A
vit.encoder.layer.1.attention.attention.value.lora_B
vit.encoder.layer.2.attention.attention.query.lora_A
vit.encoder.layer.2.attention.attention.query.lora_B
vit.encoder.layer.2.attention.attention.key.lora_A
vit.encoder.layer.2.attention.attention.key.lora_B
vit.encoder.layer.2.attention.attention.value.lora_A
vit.encoder.layer.2.attention.attention.value.lora_B
vit.encoder.layer.3.attention.attention.query.lora_A
vit.e

In [155]:
def print_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable} / Total: {total} ({100*trainable/total:.2f}%)")

# Check factorization is the only trainable component
print_trainable_params(model)

Trainable: 228874 / Total: 86027530 (0.27%)


In [156]:
# Define transforms (ViT expects 224x224 images)
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
    std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225])
])

# Download dataset
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# Create dataloaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

Files already downloaded and verified
Files already downloaded and verified


In [157]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Device: ", device)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)
criterion = nn.CrossEntropyLoss()

# Start the timer
start_time = time.time()

# Training loop
NUM_EPOCHS = 5
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    model.train()

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"\nEpoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    print(f"Validation Accuracy: {100*correct/total:.2f}%\n")

# End the timer
end_time = time.time()

# Calculate and print the total time taken
total_time = end_time - start_time
print(f"Total time taken for {NUM_EPOCHS} epochs: {total_time:.2f} seconds")

Device:  cuda
Epoch 1 | Batch 0 | Loss: 2.3661
Epoch 1 | Batch 100 | Loss: 0.5593
Epoch 1 | Batch 200 | Loss: 0.6187
Epoch 1 | Batch 300 | Loss: 1.8540
Epoch 1 | Batch 400 | Loss: 0.2300
Epoch 1 | Batch 500 | Loss: 0.1126
Epoch 1 | Batch 600 | Loss: 0.3297
Epoch 1 | Batch 700 | Loss: 0.1444
Epoch 1 | Batch 800 | Loss: 0.0469
Epoch 1 | Batch 900 | Loss: 0.6503
Epoch 1 | Batch 1000 | Loss: 0.2540
Epoch 1 | Batch 1100 | Loss: 0.0395
Epoch 1 | Batch 1200 | Loss: 0.3096
Epoch 1 | Batch 1300 | Loss: 0.0087
Epoch 1 | Batch 1400 | Loss: 0.1099
Epoch 1 | Batch 1500 | Loss: 0.1275
Epoch 1 | Batch 1600 | Loss: 0.7858
Epoch 1 | Batch 1700 | Loss: 0.1283
Epoch 1 | Batch 1800 | Loss: 0.6508
Epoch 1 | Batch 1900 | Loss: 0.0911
Epoch 1 | Batch 2000 | Loss: 0.0113
Epoch 1 | Batch 2100 | Loss: 0.0403
Epoch 1 | Batch 2200 | Loss: 0.0026
Epoch 1 | Batch 2300 | Loss: 0.1787
Epoch 1 | Batch 2400 | Loss: 0.0053
Epoch 1 | Batch 2500 | Loss: 0.0198
Epoch 1 | Batch 2600 | Loss: 0.0089
Epoch 1 | Batch 2700 | Los

KeyboardInterrupt: 

In [ ]:
######## Saving checkpoint..

checkpoint = {
    'epoch': epoch + 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': total_loss,
    'batch_idx': batch_idx,
}
torch.save(checkpoint, 'checkpoint_5_epochs_.pth')

In [13]:
######## Loading model

model_loaded = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

checkpoint = torch.load('checkpoint_5_epochs_.pth')

model_loaded = replace_attention_layers(model_loaded)
change_classifier(model_loaded)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
freeze_params_except_lora_and_classifier(model_loaded)

optimizer_loaded = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_loaded.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)
optimizer_loaded.load_state_dict(checkpoint['optimizer_state_dict'])

In [16]:
##### Inference test

device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


model_loaded.eval()
model_loaded.to(device)

image = Image.open("image.jpeg")

# Preprocess the image
input_tensor = test_transform(image).unsqueeze(0).to(device)  # Add batch dimension

# Perform inference
with torch.no_grad():
    outputs = model_loaded(input_tensor)

# Get the predicted class
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()

# Map the index to a class label (CIFAR-10 classes)
class_labels = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
predicted_class = class_labels[predicted_class_idx]

print(f"Predicted class: {predicted_class}")

# Get class probabilities
probs = torch.nn.functional.softmax(logits, dim=-1)
for i, prob in enumerate(probs[0]):
    print(f"Class {class_labels[i]}: {prob.item():.4f}")

Predicted class: horse
Class airplane: 0.0059
Class automobile: 0.0000
Class bird: 0.1241
Class cat: 0.0002
Class deer: 0.0960
Class dog: 0.0000
Class frog: 0.0000
Class horse: 0.7737
Class ship: 0.0000
Class truck: 0.0000


In [32]:
######## Fine tuning loop to continue the fine tuning..

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)
model_loaded.to(device)

optimizer_loaded = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_loaded.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)
criterion = nn.CrossEntropyLoss()

print("Re-starting the fine tuning process..")

# Training loop
start_epoch = checkpoint['epoch']
NUM_EPOCHS = 5

print(f"Resuming fine tuning starting from epoch {start_epoch + 1}...")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    model_loaded.train()

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_loaded.zero_grad()
        outputs = model_loaded(inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer_loaded.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {start_epoch + epoch + 1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    # Validation
    model_loaded.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_loaded(inputs)
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"\nEpoch {start_epoch + epoch + 1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    print(f"Validation Accuracy: {100*correct/total:.2f}%\n")

Device:  cuda
Re-starting the fine tuning process..
Resuming fine tuning starting from epoch 6...
Epoch 6 | Batch 0 | Loss: 0.3233
Epoch 6 | Batch 100 | Loss: 0.1678
Epoch 6 | Batch 200 | Loss: 2.3203
Epoch 6 | Batch 300 | Loss: 0.5157
Epoch 6 | Batch 400 | Loss: 1.5852
Epoch 6 | Batch 500 | Loss: 0.9096
Epoch 6 | Batch 600 | Loss: 0.2035
Epoch 6 | Batch 700 | Loss: 0.2872
Epoch 6 | Batch 800 | Loss: 0.9209
Epoch 6 | Batch 900 | Loss: 0.3357
Epoch 6 | Batch 1000 | Loss: 0.1326
Epoch 6 | Batch 1100 | Loss: 0.2711
Epoch 6 | Batch 1200 | Loss: 0.8132
Epoch 6 | Batch 1300 | Loss: 0.2016
Epoch 6 | Batch 1400 | Loss: 1.0649
Epoch 6 | Batch 1500 | Loss: 0.1510
Epoch 6 | Batch 1600 | Loss: 0.7029
Epoch 6 | Batch 1700 | Loss: 0.2303
Epoch 6 | Batch 1800 | Loss: 0.8210
Epoch 6 | Batch 1900 | Loss: 2.2247
Epoch 6 | Batch 2000 | Loss: 0.3992
Epoch 6 | Batch 2100 | Loss: 2.4234
Epoch 6 | Batch 2200 | Loss: 1.1490
Epoch 6 | Batch 2300 | Loss: 0.2097
Epoch 6 | Batch 2400 | Loss: 0.1064
Epoch 6 | Batc

In [33]:
######## Saving fined tuned checkpoint..

checkpoint_ft = {
    'epoch': epoch + 1,
    'model_state_dict': model_loaded.state_dict(),
    'optimizer_state_dict': optimizer_loaded.state_dict(),
    'loss': total_loss,
    'batch_idx': batch_idx,
}
torch.save(checkpoint, 'checkpoint_5_epochs__ft.pth')